# Data Cleaning


In [45]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
# Load the data
df = pd.read_csv('../data/hmeq.csv')

# Create a copy for cleaning 
df_cleaning = df.copy()

In [47]:
# Understand the missing pattern
missing_summary = pd.DataFrame({
    "Missing_count" : df.isnull().sum(),
    "Missing_percentage" : (df.isnull().sum() / len(df) * 100).round(2),
    "Data_type" : df.dtypes,
})
missing_summary = missing_summary.sort_values("Missing_percentage", ascending=False)
missing_summary


,Missing_count,Missing_percentage,Data_type
DEBTINC,1267,21.26,float64
DEROG,708,11.88,float64
DELINQ,580,9.73,float64
MORTDUE,518,8.69,float64
YOJ,515,8.64,float64
NINQ,510,8.56,float64
CLAGE,308,5.17,float64
JOB,279,4.68,str
REASON,252,4.23,str
CLNO,222,3.72,float64


In [48]:
# Create flags for missing values 
df_cleaning['DEBTINC_missing'] = df['DEBTINC'].isnull().astype(int)
df_cleaning['MORTDUE_missing'] = df['MORTDUE'].isnull().astype(int)
df_cleaning['VALUE_missing'] = df['VALUE'].isnull().astype(int)
df_cleaning.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,NaN,1,0,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,NaN,1,0,0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,NaN,1,0,0
3,1,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,NaN,1,0,0


In [49]:
# strategy for DEBTINC median imputation by JOB category
# first clean JOB
df_cleaning['JOB'] = df_cleaning['JOB'].fillna('Unknown')

# Impute DEBTINC by job cat
debtinc_by_job = df_cleaning.groupby("JOB")["DEBTINC"].median().round(2)
debtinc_by_job

JOB
Mgr        35.66
Office     36.16
Other      35.25
ProfExe    33.38
Sales      35.76
Self       34.83
Unknown    30.31
Name: DEBTINC, dtype: float64

In [50]:
df_cleaning['DEBTINC'] = df_cleaning.groupby('JOB')['DEBTINC'].transform(
    lambda x: x.fillna(x.median())
)
df_cleaning.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,35.247328,1,0,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,35.247328,1,0,0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,35.247328,1,0,0
3,1,1500,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,30.311902,1,1,1
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,36.158718,1,0,0


In [ ]:
# MORTDUE and VALUE often missing together 
# Strategy median imputation 

print(f"   MORTDUE median: {df_cleaning['MORTDUE'].median()}")
print(f"   VALUE median: {df_cleaning['VALUE'].median()}")

df_cleaning['MORTDUE'] = df_cleaning['MORTDUE'].fillna(df_cleaning['MORTDUE'].median())
df_cleaning['VALUE'] = df_cleaning['VALUE'].fillna(df_cleaning['VALUE'].median())



   MORTDUE median: 65019.0
   VALUE median: 89235.5


,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,DEBTINC_missing,MORTDUE_missing,VALUE_missing
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,35.247328,1,0,0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,35.247328,1,0,0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,35.247328,1,0,0
3,1,1500,65019.0,89235.5,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,30.311902,1,1,1
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,36.158718,1,0,0


In [57]:
# YOJ strategy median
df_cleaning['YOJ'] = df_cleaning['YOJ'].fillna(df_cleaning['YOJ'].median())

In [56]:
# REASONS categorical 
df_cleaning['REASON'] = df_cleaning['REASON'].fillna(df_cleaning['REASON'].mode()[0])


In [61]:
# Credit Behavior Variables (DEROG, DELINQ, CLAGE, NINQ, CLNO) median
df_cleaning['DEROG'] = df_cleaning['DEROG'].fillna(df_cleaning['DEROG'].median())
df_cleaning['DELINQ'] = df_cleaning['DELINQ'].fillna(df_cleaning['DELINQ'].median())
df_cleaning['CLAGE'] = df_cleaning['CLAGE'].fillna(df_cleaning['CLAGE'].median())
df_cleaning['NINQ'] = df_cleaning['NINQ'].fillna(df_cleaning['NINQ'].median())
df_cleaning['CLNO'] = df_cleaning['CLNO'].fillna(df_cleaning['CLNO'].median())

In [63]:
# verify missing data remaining 
remaining_missing = df_cleaning.isnull().sum().sum()
if remaining_missing == 0:
    print("All missing values successfully handled")
else:
    print(f"Remaining data to clea: {remaining_missing}")

All missing values successfully handled


In [64]:
# Save cleaned data
df_cleaning.to_csv('../data/hmeq_cleaned.csv', index=False)